In [1]:
import random
import math
import pandas as pd
import numpy as np
import time


from preferences import prefs
from user import user_prefs, experiments

startPoint = [59.927085, 30.317504]
endPoint = [59.935408, 30.327108]

In [2]:
df = pd.read_csv('data/places.csv')

In [3]:
def point_interest_score(point_row):
    score = 0
    for category, tags in prefs.items():
        weight = user_prefs[category]
        for tag in tags:
            if tag in point_row.index and point_row[tag]:
                score += weight

    return score


# =========================================================
# Бонус за разнообразие маршрута
# =========================================================

def diversity_bonus(route, df):
    """
    Поощряем разнообразие категорий.
    """

    used_categories = set()

    for idx in route:

        point = df.iloc[idx]

        for category_name, tags in tag_groups.items():

            for tag in tags:

                if tag in point and point[tag]:
                    used_categories.add(category_name)

    return len(used_categories) * 15


# =========================================================
# Штраф за хаотичность маршрута
# =========================================================

def route_smoothness_penalty(route, df):

    if len(route) < 3:
        return 0

    penalty = 0

    prev_direction = None

    points = []

    # START
    points.append(startPoint)

    # ROUTE
    for idx in route:

        point = df.iloc[idx]

        points.append([
            float(point["lat"]),
            float(point["lon"])
        ])

    # END
    points.append(endPoint)

    # -----------------------------------------------------
    # Анализируем изменения направления
    # -----------------------------------------------------

    for i in range(len(points) - 1):

        p1 = points[i]
        p2 = points[i + 1]

        direction = (
            p2[0] - p1[0],
            p2[1] - p1[1]
        )

        if prev_direction is not None:

            dot = (
                direction[0] * prev_direction[0]
                + direction[1] * prev_direction[1]
            )

            # Если движение назад
            if dot < 0:
                penalty += 50

        prev_direction = direction

    return penalty


# =========================================================
# Основная fitness-функция
# =========================================================

def fitness_function(route, df):

    total_interest = 0

    for idx in route:

        point = df.iloc[idx]

        total_interest += point_interest_score(point)

    distance = route_distance(route, df)

    fitness = (
        total_interest * 20
        - distance * 0.03
    )

    return fitness

In [4]:
POPULATION_SIZE = 100
GENERATIONS = 100

MIN_ROUTE_POINTS = 2
MAX_ROUTE_POINTS = 15

MUTATION_RATE = 0.2
TOURNAMENT_SIZE = 5


# Особь:
# {
#     "route": [индексы точек из df],
#     "fitness": число
# }
#
# Полный маршрут:


# Расстояние между точками в метрах
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)

    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1)
        * math.cos(phi2)
        * math.sin(dlambda / 2) ** 2
    )

    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))


# Длина маршрута
def route_distance(route, df):

    total_distance = 0

    prev_lat, prev_lon = startPoint

    for idx in route:
        point = df.iloc[idx]

        total_distance += haversine(
            prev_lat,
            prev_lon,
            point["lat"],
            point["lon"]
        )

        prev_lat = point["lat"]
        prev_lon = point["lon"]

    total_distance += haversine(
        prev_lat,
        prev_lon,
        endPoint[0],
        endPoint[1]
    )

    return total_distance


# Генерация случайной особи
def create_individual(df):

    route_size = random.randint(
        MIN_ROUTE_POINTS,
        MAX_ROUTE_POINTS
    )

    route = random.sample(
        range(len(df)),
        route_size
    )

    return {
        "route": route,
        "fitness": None
    }


# Создание популяции
def create_population(df):
    return [
        create_individual(df)
        for _ in range(POPULATION_SIZE)
    ]


# Оценка популяции
def evaluate_population(population, df):

    for individual in population:
        individual["fitness"] = fitness_function(
            individual["route"],
            df
        )


# Турнир
def tournament_selection(population):

    tournament = random.sample(
        population,
        TOURNAMENT_SIZE
    )

    tournament.sort(
        key=lambda x: x["fitness"],
        reverse=True
    )

    return tournament[0]


# Кроссовер
def crossover(parent1, parent2):

    route1 = parent1["route"]
    route2 = parent2["route"]

    if len(route1) < 2 or len(route2) < 2:
        return {
            "route": route1.copy(),
            "fitness": None
        }

    cut1 = random.randint(1, len(route1) - 1)
    cut2 = random.randint(1, len(route2) - 1)

    child_route = route1[:cut1]

    for point in route2[cut2:]:
        if point not in child_route:
            child_route.append(point)

    return {
        "route": child_route,
        "fitness": None
    }


# Мутация
def mutate(individual, df):

    route = individual["route"][:]

    if random.random() > MUTATION_RATE:
        return individual

    mutation_type = random.choice([
        "swap",
        "remove",
        "add",
        "shuffle"
    ])

    # -----------------------------------------
    # Swap
    # -----------------------------------------
    if mutation_type == "swap" and len(route) >= 2:

        i, j = random.sample(range(len(route)), 2)
        route[i], route[j] = route[j], route[i]

    # -----------------------------------------
    # Remove
    # -----------------------------------------
    elif mutation_type == "remove":

        if len(route) > MIN_ROUTE_POINTS:
            idx = random.randint(0, len(route) - 1)
            route.pop(idx)

    # -----------------------------------------
    # Add
    # -----------------------------------------
    elif mutation_type == "add":

        if len(route) < MAX_ROUTE_POINTS:

            available = list(
                set(range(len(df))) - set(route)
            )

            if available:

                count_to_add = random.randint(1, 3)

                for _ in range(count_to_add):

                    if not available:
                        break

                    new_point = random.choice(available)
                    available.remove(new_point)

                    insert_pos = random.randint(
                        0,
                        len(route)
                    )

                    route.insert(insert_pos, new_point)

    # -----------------------------------------
    # Shuffle
    # -----------------------------------------
    elif mutation_type == "shuffle":

        random.shuffle(route)

    individual["route"] = route
    individual["fitness"] = None

    return individual


# Создание нового поколения
def create_next_generation(population, df):

    new_population = []

    # Элитизм
    population.sort(
        key=lambda x: x["fitness"],
        reverse=True
    )

    elite = population[:5]

    new_population.extend(elite)

    # Остальная популяция
    while len(new_population) < POPULATION_SIZE:

        parent1 = tournament_selection(population)
        parent2 = tournament_selection(population)

        child = crossover(parent1, parent2)

        child = mutate(child, df)

        new_population.append(child)

    return new_population


# Генетический алгоритм
def genetic_algorithm(df):

    population = create_population(df)

    for generation in range(GENERATIONS):

        evaluate_population(population, df)

        best = max(
            population,
            key=lambda x: x["fitness"]
        )

        print(
            f"Generation {generation} | "
            f"Best fitness: {best['fitness']:.10f} | "
            f"Route length: {len(best['route'])}"
        )

        population = create_next_generation(
            population,
            df
        )

    evaluate_population(population, df)

    best = max(
        population,
        key=lambda x: x["fitness"]
    )

    return best


# Полный маршрут
def project_point_to_line(point, start, end):
    """
    Проекция точки на линию START -> END.
    Возвращает параметр t:
    0   = начало линии
    1   = конец линии
    >1  = дальше конца
    <0  = до начала
    """

    px, py = point
    sx, sy = start
    ex, ey = end

    line_vec = np.array([ex - sx, ey - sy])
    point_vec = np.array([px - sx, py - sy])

    line_len_sq = np.dot(line_vec, line_vec)

    if line_len_sq == 0:
        return 0

    t = np.dot(point_vec, line_vec) / line_len_sq

    return t


def build_full_route(best_individual, df):

    route_points = []

    # ==========================================
    # Получаем точки маршрута
    # ==========================================

    points = []

    for idx in best_individual["route"]:

        point = df.iloc[idx]

        lat = float(point["lat"])
        lon = float(point["lon"])

        # --------------------------------------
        # Позиция вдоль линии START -> END
        # --------------------------------------

        t = project_point_to_line(
            [lat, lon],
            startPoint,
            endPoint
        )

        points.append({
            "name": point["name"],
            "lat": lat,
            "lon": lon,
            "t": t
        })

    # ==========================================
    # Сортировка по географическому порядку
    # ==========================================

    points.sort(key=lambda x: x["t"])

    # ==========================================
    # Финальный маршрут
    # ==========================================

    route_points.append({
        "name": "START",
        "lat": startPoint[0],
        "lon": startPoint[1]
    })

    route_points.extend(points)

    route_points.append({
        "name": "END",
        "lat": endPoint[0],
        "lon": endPoint[1]
    })

    return route_points

    route_points = []

    route_points.append({
        "name": "START",
        "lat": startPoint[0],
        "lon": startPoint[1]
    })

    for idx in best_individual["route"]:

        point = df.iloc[idx]

        route_points.append({
            "name": point["name"],
            "lat": point["lat"],
            "lon": point["lon"]
        })

    route_points.append({
        "name": "END",
        "lat": endPoint[0],
        "lon": endPoint[1]
    })

    return route_points

# Ссылка на яндекс карты
def generate_yandex_maps_url(route_points):
    """
    route_points:
    [
        {"name": "...", "lat": ..., "lon": ...},
        ...
    ]
    """

    rtext = "~".join(
        f"{point['lat']},{point['lon']}"
        for point in route_points
    )

    yandex_url = (
        f"https://yandex.ru/maps/"
        f"?mode=routes"
        f"&rtext={rtext}"
        f"&rtt=pd"
    )

    return yandex_url

# Запуск

best_individual = genetic_algorithm(df)

final_route = build_full_route(
    best_individual,
    df
)


# =========================================================
# Только список названий мест
# =========================================================

print("\n==============================")
print("СПИСОК МЕСТ")
print("==============================\n")

places_names = []

for point in final_route:

    # START / END можно исключить
    if point["name"] not in ["START", "END"]:
        places_names.append(point["name"])

for i, name in enumerate(places_names):

    print(f"{i + 1}. {name}")

# =========================================================
# Ссылка на Яндекс.Карты
# =========================================================

yandex_url = generate_yandex_maps_url(
    final_route
)

print("\n==============================")
print("ССЫЛКА НА МАРШРУТ")
print("==============================\n")

print(yandex_url)

Generation 0 | Best fitness: -139.5706809847 | Route length: 2
Generation 1 | Best fitness: 147.2672439820 | Route length: 3
Generation 2 | Best fitness: 147.2672439820 | Route length: 3
Generation 3 | Best fitness: 172.2933286999 | Route length: 3
Generation 4 | Best fitness: 178.8310757598 | Route length: 3
Generation 5 | Best fitness: 264.7949060475 | Route length: 4
Generation 6 | Best fitness: 281.1020909778 | Route length: 5
Generation 7 | Best fitness: 285.2745714944 | Route length: 4
Generation 8 | Best fitness: 388.3609233993 | Route length: 6
Generation 9 | Best fitness: 394.6993308285 | Route length: 6
Generation 10 | Best fitness: 423.7077801106 | Route length: 7
Generation 11 | Best fitness: 492.8957169037 | Route length: 7
Generation 12 | Best fitness: 492.8957169037 | Route length: 7
Generation 13 | Best fitness: 492.8957169037 | Route length: 7
Generation 14 | Best fitness: 492.8957169037 | Route length: 7
Generation 15 | Best fitness: 508.2202725943 | Route length: 7
G

In [7]:
def route_interest_score(route, df):
    total = 0

    for idx in route:
        point = df.iloc[idx]
        total += point_interest_score(point)

    return total

def run_experiment(df, user_prefs, experiment_name="exp"):

    start_time = time.time()

    best_individual = genetic_algorithm(df)

    final_route = build_full_route(best_individual, df)

    end_time = time.time()

    # -------------------------------
    # метрики
    # -------------------------------

    interest = route_interest_score(best_individual["route"], df)
    distance = route_distance(best_individual["route"], df)
    duration = end_time - start_time

    yandex_url = generate_yandex_maps_url(final_route)

    route_text = route_to_names(final_route)

    return {
        "experiment": experiment_name,
        "interest_score": interest,
        "distance": distance,
        "time_sec": duration,
        "route_length": len(best_individual["route"]),
        "route_text": route_text,
        "yandex_url": yandex_url
    }


def run_multiple_experiments(df, experiments):

    results = []

    for exp in experiments:

        print(f"\nRunning: {exp['name']}")

        result = run_experiment(
            df=df,
            user_prefs=exp["user_prefs"],
            experiment_name=exp["name"]
        )

        results.append(result)

        print(f"Done: {exp['name']} | time: {result['time_sec']:.2f}s")

    return results

def route_to_names(final_route):

    names = []

    for point in final_route:

        if point["name"] not in ["START", "END"]:
            names.append(point["name"])

    return " -> ".join(names)

def save_results(results, filename="ga_experiments.csv"):

    rows = []

    for r in results:

        rows.append({
            "experiment": r["experiment"],
            "interest_score": r["interest_score"],
            "distance": r["distance"],
            "time_sec": r["time_sec"],
            "route_length": r["route_length"],
            "route_text": r["route_text"],
            "yandex_url": r["yandex_url"]
        })

    df_results = pd.DataFrame(rows)
    df_results.to_csv(filename, index=False)

    print(f"\nSaved to {filename}")

results = run_multiple_experiments(df, experiments)

save_results(results)


Running: user_1_military_focus
Generation 0 | Best fitness: -116.2767410665 | Route length: 3
Generation 1 | Best fitness: -60.6263908039 | Route length: 2
Generation 2 | Best fitness: -16.7086718543 | Route length: 2
Generation 3 | Best fitness: 106.7559136251 | Route length: 2
Generation 4 | Best fitness: 220.5366788396 | Route length: 3
Generation 5 | Best fitness: 221.2366058314 | Route length: 3
Generation 6 | Best fitness: 221.2366058314 | Route length: 3
Generation 7 | Best fitness: 221.2366058314 | Route length: 3
Generation 8 | Best fitness: 226.1279574231 | Route length: 3
Generation 9 | Best fitness: 226.1279574231 | Route length: 3
Generation 10 | Best fitness: 226.1279574231 | Route length: 3
Generation 11 | Best fitness: 226.6637073416 | Route length: 3
Generation 12 | Best fitness: 226.6637073416 | Route length: 3
Generation 13 | Best fitness: 245.2021860006 | Route length: 4
Generation 14 | Best fitness: 247.2448528072 | Route length: 4
Generation 15 | Best fitness: 24